# Milestone 6 Lab, notebook 2 of 8: Corrections and incremental runs

## Where we are

| Notebook | What you do |
|---|---|
| 1. Late and corrected files, by hand | Work out by hand what three late files should do to the figures |
| **2. Corrections and incremental runs** (you are here) | Load the late files, recalculate only what they touch, and prove it equals a full rebuild |
| 3. The review desk | Decide the findings that need review, and watch the next run obey |
| 4. Run health and restatements | Read a run's health, and ask the analyst why a reported figure changed |
| 5. A judge with a reference answer | Test a judge on explanations whose grade you know, then use it |
| 6. The end-to-end evaluation | Score the whole system from files to answers, on two question sets |
| 7. The improver | An agent proposes a change to the analyst, you approve, and code keeps it or rolls it back |
| 8. The whole system | Run everything from files to answer, and look back over six milestones |

Model requests in this notebook: **0**. A run of this lab with its default small sets costs about 10 US cents.

## What this notebook is for

In notebook 1 you worked out what three late files should do. Here the system does it:

1. The loader accepts the **correction**, and loads nothing twice.
2. The pipeline recalculates **only the months** the files touch.
3. You prove that this equals recalculating everything.
4. The run history shows which figures were **restated**, and which files caused it.

No AI is used in this notebook.

> Run notebook 1 first. This notebook builds on the folder it made.

In [ ]:
from dataclasses import replace

import pandas as pd
from lab_helpers import LAB_E2E, load_result, profit_of, show_doc

from retail_profit_stack.e2e import build_close

workspace = build_close(LAB_E2E)
restated_key = load_result("answer_key_restated", required_from="notebook 1")

The first cell rebuilt the system to month-end close, so you can run this notebook again from a clean start.

## Step 1: The design

In [ ]:
show_doc("HLD.md", "5. Late data, corrections, and incremental runs")

## Step 2: Load the late files

The loader is milestone 5's `load_batch`. We point it at the late folder. `replace` makes a copy of the lab's folders with one of them changed.

In [ ]:
from retail_profit_stack.ingest.dispatch import load_batch

late_places = replace(workspace.places, landing=workspace.late)

results = load_batch(places=late_places)

for item in results:
    print(f"{item['outcome']:8} {item['file_name']:34} {item['reason']}")

All three loaded. Read the reason on the shipments file: **a correction**. In milestone 5 the same file would have been rejected.

The rule changed in one place. A file whose name was loaded before, with different content, is now treated as the sender's own fix.

In [ ]:
from retail_profit_stack.ingest import ledger

print("Outcomes:", ledger.outcome_counts(workspace.places.state_file))
print("Replaced:", ledger.files_replaced(workspace.places.state_file))

The load record keeps a note of the version that was replaced. And there are still nine shipment pieces, not ten: the corrected file took the place of the old one.

In [ ]:
pieces = sorted(piece.name for piece in (workspace.places.loaded / "shipments").glob("*.parquet"))

print(len(pieces), "shipment pieces")

Loading is still safe to repeat. Load the late files again.

In [ ]:
again = load_batch(places=late_places)

print({item["outcome"] for item in again})

## Step 3: Which months changed? Code works it out

In notebook 1 you read the months off the files. The pipeline does the same. A run's record now lists every file behind each table, with its fingerprint. Comparing that list with the run before shows which files are new or changed.

In [ ]:
from retail_profit_stack.lineage.runs import input_fingerprints, read_run_record
from retail_profit_stack.transform.pipeline import load_sources

previous = read_run_record("RUN-0002", workspace.warehouse)
tables = load_sources(workspace.places.loaded)
row_counts = {name: table.num_rows for name, table in tables.items()}
now = input_fingerprints(workspace.places.loaded, row_counts)

for table in now:
    before = previous["inputs"][table]["pieces"]
    for piece, fingerprint in now[table]["pieces"].items():
        if piece not in before:
            print("new      ", piece)
        elif before[piece] != fingerprint:
            print("corrected", piece)

Three files, found by comparing fingerprints. The pipeline then reads the dates inside those three files to get the months.

## Step 4: An incremental run

`incremental=True` asks the pipeline to recalculate only the months that changed files touch, and to copy every other month from the run before.

In [ ]:
from retail_profit_stack.transform.pipeline import run_pipeline

late_run = run_pipeline(
    "sql",
    source_dir=workspace.places.loaded,
    warehouse_dir=workspace.warehouse,
    note="Late and corrected files arrived after month-end close",
    incremental=True,
)

print("Run:   ", late_run.run_id)
print("Mode:  ", late_run.record["mode"])
print("Months:", late_run.record["months_recalculated"])

September and October, as you worked out by hand in notebook 1.

## Step 5: The proof

An incremental run is only worth having if it gives **exactly** what a full run gives. So run the pipeline again in full, saving nothing, and compare every table.

In [ ]:
full_run = run_pipeline("sql", source_dir=workspace.places.loaded, warehouse_dir=None)

for name, table in late_run.tables.items():
    same = table.equals(full_run.tables[name])
    print(f"{name:24} {table.num_rows} rows   same as a full rebuild: {same}")

Every table is identical. Why is that guaranteed, and not luck? Because every figure depends only on records dated in its own month. A September refund cannot change an October figure.

One thing is **not** incremental: the checks. They still look at all the records, because a duplicate can sit in a different month from its copy. They are fast, so nothing is lost.

## Step 6: Check against your answer key

In [ ]:
names = {"P012 2026-09": "Fleece pullover, September", "P001 2026-10": "Trail water bottle, October",
         "P007 2026-10": "Camp stove, October"}

rows = []
for key, name in names.items():
    product_id, month = key.split()
    calculated = profit_of("RUN-0003", product_id, month, workspace.warehouse)
    rows.append({
        "figure": name,
        "at close": restated_key[key]["before"],
        "by hand (notebook 1)": restated_key[key]["after"],
        "calculated": calculated,
        "same?": "yes" if calculated == restated_key[key]["after"] else "NO",
    })

pd.DataFrame(rows)

## Step 7: The restatement is on record

Nothing was overwritten. `RUN-0002` still holds the figures as they were reported at close, and `RUN-0003` holds the new ones. Milestone 4's `compare_runs` now also names the **files** that differ.

In [ ]:
from retail_profit_stack.lineage.compare import compare_runs

comparison = compare_runs("RUN-0002", "RUN-0003", workspace.warehouse)

print("Files added:    ", comparison["files_added"])
print("Files corrected:", comparison["files_corrected"])
print("Checks changed: ", comparison["checks_changed"])

In [ ]:
figures = pd.DataFrame(comparison["figures_changed"])
figures[figures["metric"] == "contribution_profit"]

Three restated figures, and the files that caused them. The camp stove's `change` is empty, because "unavailable" has no amount to subtract from. The missing-cost check, which needed review since milestone 3, now passes.

## Step 8: An old run can still be traced

Here is a subtle problem. The shipments records changed. `RUN-0002`'s lineage points at records by their row. If we looked those rows up in today's records, we might read the corrected values, and the old figure would no longer add up.

So each run now keeps a copy of the records it used.

In [ ]:
run_folder = workspace.warehouse / "runs" / "RUN-0002"

print(sorted(item.name for item in (run_folder / "sources").iterdir()))

In [ ]:
from retail_profit_stack import tools
from retail_profit_stack.store import Store

at_close = Store(source_dir=workspace.places.loaded, warehouse_dir=workspace.warehouse,
                 run_id="RUN-0002", state_file=workspace.places.state_file)

evidence = tools.get_evidence("2026-10", "P001", store=at_close)

print("Run:", evidence.run_id, "  records behind the figure:", evidence.total_records)
print(profit_of("RUN-0002", "P001", "2026-10", workspace.warehouse))

`RUN-0002` still answers from the records as they were: 754 records and $5,156.94, the figure you traced by hand in milestone 4.

## Step 9: A run with nothing new

What does an incremental run do when no file has arrived?

In [ ]:
idle_run = run_pipeline("sql", source_dir=workspace.places.loaded, warehouse_dir=workspace.warehouse,
                        note="Nothing new", incremental=True)

print("Mode:", idle_run.record["mode"], "  months recalculated:", idle_run.record["months_recalculated"])
print("Figures changed since the run before:", len(compare_runs("RUN-0003", idle_run.run_id, workspace.warehouse)["figures_changed"]))

No month is recalculated, and nothing changes.

## What we now know

- A file with a loaded name and new content is a **correction**. It replaces the earlier version, and the load record notes what was replaced.
- The pipeline finds changed files by their **fingerprints**, and reads the months from the records inside them.
- An **incremental run** recalculates only those months. We proved it equals a full rebuild, table by table.
- A **restatement** is never silent: the run before keeps the old figure, and the comparison names the files.
- Each run keeps its own copy of the records, so an old figure can still be traced after a correction.

**Next:** notebook 3. Three findings still say "needs review". Someone has to decide them.